# Projeto: Previsao de Demanda de Vendas - Rossmann

## Notebook 02 - Modelagem e Comparacao de Algoritmos

### Objetivos deste Notebook
1. **Treinar multiplos modelos** de regressao
2. **Comparar desempenho** usando metricas apropriadas
3. **Analisar importancia** das features
4. **Otimizar hiperparametros** do melhor modelo
5. **Visualizar resultados** de forma clara e profissional

### Modelos que serao comparados:
- Regressao Linear (baseline simples)
- Random Forest Regressor (ensemble baseado em arvores)
- XGBoost Regressor (gradient boosting otimizado)
- LightGBM (gradient boosting rapido)

### Metricas de Avaliacao:
- **RMSE** (Root Mean Squared Error): penaliza erros grandes
- **MAE** (Mean Absolute Error): erro medio absoluto
- **MAPE** (Mean Absolute Percentage Error): erro percentual
- **R²** (R-squared): proporcao de variancia explicada

In [1]:
# IMPORTACOES E CONFIGURACOES

import pandas as pd
import numpy as np
import json
import time

# Visualizacao
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import lightgbm as lgb

# Configuracoes
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.float_format', '{:.2f}'.format)

print("Bibliotecas importadas com sucesso!\n")
print("Versoes das bibliotecas principais:")
print(f"   - XGBoost: {xgb.__version__}")
print(f"   - LightGBM: {lgb.__version__}")
print(f"   - Pandas: {pd.__version__}")
print(f"   - NumPy: {np.__version__}")

Bibliotecas importadas com sucesso!

Versoes das bibliotecas principais:
   - XGBoost: 3.1.3
   - LightGBM: 4.6.0
   - Pandas: 2.3.3
   - NumPy: 2.4.1


In [2]:
# CARREGAMENTO DOS DADOS PROCESSADOS

print("Carregando dados processados...\n")

train_data = pd.read_csv('../data/processed/train_processed.csv', parse_dates=['Date'])
test_data = pd.read_csv('../data/processed/test_processed.csv', parse_dates=['Date'])

# Carregar lista de features
with open('../data/processed/features.json', 'r') as f:
    features_dict = json.load(f)

all_features = features_dict['all_features']
cat_features = features_dict['cat_features']
num_features = features_dict['num_features']
target = features_dict['target']

print(f"Dados carregados:")
print(f"   Train: {len(train_data):,} registros")
print(f"   Test: {len(test_data):,} registros")
print(f"   Features: {len(all_features)}")
print(f"   Target: {target}")

Carregando dados processados...

Dados carregados:
   Train: 802,014 registros
   Test: 199,585 registros
   Features: 30
   Target: Sales


In [3]:
# PRE-PROCESSAMENTO PARA MODELAGEM

print("Preparando dados para modelagem...\n")

# Criar copias
train_ml = train_data.copy()
test_ml = test_data.copy()

# Label Encoding para variaveis categoricas
label_encoders = {}

for col in cat_features:
    le = LabelEncoder()
    
    # Fit no conjunto completo
    all_values = pd.concat([train_ml[col], test_ml[col]]).astype(str)
    le.fit(all_values)
    
    # Transform
    train_ml[col] = le.transform(train_ml[col].astype(str))
    test_ml[col] = le.transform(test_ml[col].astype(str))
    
    label_encoders[col] = le

print(f"{len(cat_features)} variaveis categoricas codificadas")

# Separar X e y
X_train = train_ml[all_features]
y_train = train_ml[target]

X_test = test_ml[all_features]
y_test = test_ml[target]

print(f"\nShapes finais:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test: {X_test.shape}")
print(f"\nDados prontos para treinamento!")

Preparando dados para modelagem...

10 variaveis categoricas codificadas

Shapes finais:
   X_train: (802014, 30)
   X_test: (199585, 30)

Dados prontos para treinamento!


In [4]:
# DEFINICAO DE FUNCOES DE AVALIACAO

def calculate_metrics(y_true, y_pred, model_name="Model"):
    """
    Calcula metricas de regressao completas.
    """
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    
    # MAPE (evitar divisao por zero)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    
    r2 = r2_score(y_true, y_pred)
    
    return {
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE (%)': mape,
        'R2': r2
    }

def train_and_evaluate(model, X_train, y_train, X_test, y_test, model_name):
    """
    Treina modelo, faz predicoes e retorna metricas.
    """
    print(f"\nTreinando {model_name}...")
    
    # Treino
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Predicoes
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Metricas
    metrics_train = calculate_metrics(y_train, y_pred_train, f"{model_name} (Train)")
    metrics_test = calculate_metrics(y_test, y_pred_test, f"{model_name} (Test)")
    
    print(f"Concluido em {train_time:.2f}s")
    print(f"   RMSE (test): {metrics_test['RMSE']:,.2f}")
    print(f"   R2 (test): {metrics_test['R2']:.4f}")
    
    return {
        'model': model,
        'metrics_train': metrics_train,
        'metrics_test': metrics_test,
        'y_pred_train': y_pred_train,
        'y_pred_test': y_pred_test,
        'train_time': train_time
    }

print("Funcoes de avaliacao definidas!")

Funcoes de avaliacao definidas!


In [5]:
# MODELO 1: REGRESSAO LINEAR (BASELINE)

print("="*70)
print("MODELO 1: REGRESSAO LINEAR")
print("="*70)

lr_model = LinearRegression(n_jobs=-1)

lr_results = train_and_evaluate(
    model=lr_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="Linear Regression"
)

MODELO 1: REGRESSAO LINEAR

Treinando Linear Regression...
Concluido em 0.58s
   RMSE (test): 1,379.37
   R2 (test): 0.8715


In [6]:
# MODELO 2: RANDOM FOREST REGRESSOR

print("\n" + "="*70)
print("MODELO 2: RANDOM FOREST")
print("="*70)

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42,
    verbose=0
)

rf_results = train_and_evaluate(
    model=rf_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="Random Forest"
)


MODELO 2: RANDOM FOREST

Treinando Random Forest...
Concluido em 17.00s
   RMSE (test): 808.58
   R2 (test): 0.9558


In [7]:
# MODELO 3: XGBOOST REGRESSOR

print("\n" + "="*70)
print("MODELO 3: XGBOOST")
print("="*70)

xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42,
    verbosity=0
)

xgb_results = train_and_evaluate(
    model=xgb_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost"
)


MODELO 3: XGBOOST

Treinando XGBoost...
Concluido em 3.22s
   RMSE (test): 796.13
   R2 (test): 0.9572


In [8]:
# MODELO 4: LIGHTGBM

print("\n" + "="*70)
print("MODELO 4: LIGHTGBM")
print("="*70)

lgb_model = lgb.LGBMRegressor(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42,
    verbosity=-1
)

lgb_results = train_and_evaluate(
    model=lgb_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="LightGBM"
)


MODELO 4: LIGHTGBM

Treinando LightGBM...
Concluido em 1.72s
   RMSE (test): 804.58
   R2 (test): 0.9563


In [9]:
# COMPARACAO DE MODELOS - TABELA DE RESULTADOS

print("\n" + "="*70)
print("COMPARACAO FINAL DOS MODELOS")
print("="*70 + "\n")

# Consolidar resultados
all_results = [
    lr_results,
    rf_results,
    xgb_results,
    lgb_results
]

# Criar DataFrame de comparacao
comparison_df = pd.DataFrame([
    {
        'Modelo': r['metrics_test']['Model'].replace(' (Test)', ''),
        'RMSE': r['metrics_test']['RMSE'],
        'MAE': r['metrics_test']['MAE'],
        'MAPE (%)': r['metrics_test']['MAPE (%)'],
        'R2': r['metrics_test']['R2'],
        'Tempo (s)': r['train_time']
    }
    for r in all_results
])

# Ordenar por RMSE
comparison_df = comparison_df.sort_values('RMSE').reset_index(drop=True)

print("METRICAS NO CONJUNTO DE TESTE:\n")
display(comparison_df)

# Identificar melhor modelo
best_model_name = comparison_df.iloc[0]['Modelo']
best_rmse = comparison_df.iloc[0]['RMSE']
best_r2 = comparison_df.iloc[0]['R2']

print(f"\nMELHOR MODELO: {best_model_name}")
print(f"   RMSE: {best_rmse:,.2f}")
print(f"   R2: {best_r2:.4f}")
print(f"\nInterpretacao: O modelo explica {best_r2*100:.2f}% da variancia nas vendas")


COMPARACAO FINAL DOS MODELOS

METRICAS NO CONJUNTO DE TESTE:



,Modelo,RMSE,MAE,MAPE (%),R2,Tempo (s)
0,XGBoost,796.13,520.85,8.79,0.96,3.22
1,LightGBM,804.58,526.78,8.92,0.96,1.72
2,Random Forest,808.58,501.52,8.78,0.96,17.00
3,Linear Regression,1379.37,958.33,14.35,0.87,0.58



MELHOR MODELO: XGBoost
   RMSE: 796.13
   R2: 0.9572

Interpretacao: O modelo explica 95.72% da variancia nas vendas


In [10]:
# VISUALIZACAO 1: COMPARACAO DE METRICAS

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('RMSE (quanto menor, melhor)', 
                    'R2 (quanto maior, melhor)',
                    'MAE (quanto menor, melhor)', 
                    'MAPE % (quanto menor, melhor)'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

# RMSE
fig.add_trace(
    go.Bar(x=comparison_df['Modelo'], y=comparison_df['RMSE'], 
           marker_color='indianred', name='RMSE'),
    row=1, col=1
)

# R2
fig.add_trace(
    go.Bar(x=comparison_df['Modelo'], y=comparison_df['R2'], 
           marker_color='lightseagreen', name='R2'),
    row=1, col=2
)

# MAE
fig.add_trace(
    go.Bar(x=comparison_df['Modelo'], y=comparison_df['MAE'], 
           marker_color='lightsalmon', name='MAE'),
    row=2, col=1
)

# MAPE
fig.add_trace(
    go.Bar(x=comparison_df['Modelo'], y=comparison_df['MAPE (%)'], 
           marker_color='plum', name='MAPE'),
    row=2, col=2
)

fig.update_layout(height=700, showlegend=False, 
                  title_text="Comparacao de Metricas entre Modelos")
fig.write_html("../images/model_comparison.html")
fig.show()

In [11]:
# VISUALIZACAO 2: PREDICOES VS VALORES REAIS

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[r['metrics_test']['Model'].replace(' (Test)', '') 
                    for r in all_results]
)

positions = [(1,1), (1,2), (2,1), (2,2)]

for i, (r, pos) in enumerate(zip(all_results, positions)):
    # Sample para visualizacao (max 5000 pontos)
    sample_size = min(5000, len(y_test))
    idx = np.random.choice(len(y_test), sample_size, replace=False)
    
    fig.add_trace(
        go.Scatter(
            x=y_test.iloc[idx],
            y=r['y_pred_test'][idx],
            mode='markers',
            marker=dict(size=3, opacity=0.5),
            name=r['metrics_test']['Model'].replace(' (Test)', '')
        ),
        row=pos[0], col=pos[1]
    )
    
    # Linha diagonal (predicao perfeita)
    max_val = max(y_test.max(), r['y_pred_test'].max())
    fig.add_trace(
        go.Scatter(
            x=[0, max_val],
            y=[0, max_val],
            mode='lines',
            line=dict(color='red', dash='dash'),
            showlegend=False
        ),
        row=pos[0], col=pos[1]
    )

fig.update_xaxes(title_text="Valores Reais")
fig.update_yaxes(title_text="Predicoes")
fig.update_layout(height=800, showlegend=False,
                  title_text="Predicoes vs Valores Reais (Teste)")
fig.write_html("../images/predictions_vs_actual.html")
fig.show()

print("\nINTERPRETACAO:")
print("   - Pontos proximos da linha vermelha = boas predicoes")
print("   - Dispersao = erros do modelo")
print("   - Padroes sistematicos = vies do modelo")


INTERPRETACAO:
   - Pontos proximos da linha vermelha = boas predicoes
   - Dispersao = erros do modelo
   - Padroes sistematicos = vies do modelo


In [12]:
# ANALISE DE RESIDUOS - MELHOR MODELO

# Identificar melhor modelo
best_results = min(all_results, key=lambda x: x['metrics_test']['RMSE'])
best_model_name = best_results['metrics_test']['Model'].replace(' (Test)', '')

# Calcular residuos
residuals = y_test - best_results['y_pred_test']

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Distribuicao dos Residuos', 'Residuos vs Predicoes')
)

# Histograma de residuos
fig.add_trace(
    go.Histogram(x=residuals, nbinsx=50, name='Residuos',
                 marker_color='skyblue'),
    row=1, col=1
)

# Residuos vs predicoes
sample_size = min(5000, len(y_test))
idx = np.random.choice(len(y_test), sample_size, replace=False)

fig.add_trace(
    go.Scatter(
        x=best_results['y_pred_test'][idx],
        y=residuals.iloc[idx],
        mode='markers',
        marker=dict(size=3, opacity=0.5),
        name='Residuos'
    ),
    row=1, col=2
)

# Linha zero
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=2)

fig.update_xaxes(title_text="Residuos", row=1, col=1)
fig.update_xaxes(title_text="Predicoes", row=1, col=2)
fig.update_yaxes(title_text="Frequencia", row=1, col=1)
fig.update_yaxes(title_text="Residuos", row=1, col=2)

fig.update_layout(height=400, showlegend=False,
                  title_text=f"Analise de Residuos - {best_model_name}")
fig.write_html("../images/residuals_analysis.html")
fig.show()

print(f"\nESTATISTICAS DOS RESIDUOS:")
print(f"   Media: {residuals.mean():.2f}")
print(f"   Desvio Padrao: {residuals.std():.2f}")
print(f"   Min/Max: {residuals.min():.2f} / {residuals.max():.2f}")


ESTATISTICAS DOS RESIDUOS:
   Media: 112.38
   Desvio Padrao: 788.16
   Min/Max: -12954.43 / 31098.69


In [13]:
# IMPORTANCIA DAS FEATURES

# Pegar modelo tree-based (RF, XGB ou LGB)
if 'XGBoost' in best_model_name:
    feature_importance = best_results['model'].feature_importances_
elif 'Random Forest' in best_model_name:
    feature_importance = best_results['model'].feature_importances_
elif 'LightGBM' in best_model_name:
    feature_importance = best_results['model'].feature_importances_
else:
    # Para regressao linear, usar coeficientes absolutos
    feature_importance = np.abs(best_results['model'].coef_)

# Criar DataFrame de importancia
importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False).head(20)

# Visualizar
fig = px.bar(importance_df, x='Importance', y='Feature', orientation='h',
             title=f'Top 20 Features Mais Importantes - {best_model_name}',
             labels={'Importance': 'Importancia', 'Feature': 'Feature'},
             color='Importance',
             color_continuous_scale='Viridis')

fig.update_layout(height=600, showlegend=False, yaxis={'categoryorder':'total ascending'})
fig.write_html("../images/feature_importance.html")
fig.show()

print("\nTOP 10 FEATURES MAIS IMPORTANTES:")
print("="*50)
for i, row in importance_df.head(10).iterrows():
    print(f"{i+1}. {row['Feature']}: {row['Importance']:.4f}")


TOP 10 FEATURES MAIS IMPORTANTES:
3. Open: 0.5237
5. StateHoliday: 0.1452
27. sales_lag_14: 0.1148
4. Promo: 0.0635
29. sales_rolling_30: 0.0435
2. DayOfWeek: 0.0299
25. sales_lag_1: 0.0148
28. sales_rolling_7: 0.0140
24. IsMonthEnd: 0.0090
22. IsWeekend: 0.0078


In [14]:
# COMPARACAO TREINO VS TESTE (OVERFITTING)

train_test_comparison = pd.DataFrame([
    {
        'Modelo': r['metrics_test']['Model'].replace(' (Test)', ''),
        'RMSE Train': r['metrics_train']['RMSE'],
        'RMSE Test': r['metrics_test']['RMSE'],
        'Gap (%)': ((r['metrics_test']['RMSE'] - r['metrics_train']['RMSE']) / r['metrics_train']['RMSE'] * 100)
    }
    for r in all_results
])

fig = go.Figure()

fig.add_trace(go.Bar(
    x=train_test_comparison['Modelo'],
    y=train_test_comparison['RMSE Train'],
    name='Train',
    marker_color='lightblue'
))

fig.add_trace(go.Bar(
    x=train_test_comparison['Modelo'],
    y=train_test_comparison['RMSE Test'],
    name='Test',
    marker_color='coral'
))

fig.update_layout(
    title='RMSE: Treino vs Teste (Analise de Overfitting)',
    xaxis_title='Modelo',
    yaxis_title='RMSE',
    barmode='group',
    height=500
)
fig.write_html("../images/train_vs_test.html")
fig.show()

print("\nANALISE DE OVERFITTING:")
print("="*70)
display(train_test_comparison)
print("\nInterpretacao:")
print("- Gap < 10%: modelo generaliza bem")
print("- Gap 10-20%: overfitting leve")
print("- Gap > 20%: overfitting significativo")


ANALISE DE OVERFITTING:


,Modelo,RMSE Train,RMSE Test,Gap (%)
0,Linear Regression,1436.78,1379.37,-4.00
1,Random Forest,601.52,808.58,34.42
2,XGBoost,612.64,796.13,29.95
3,LightGBM,708.53,804.58,13.56



Interpretacao:
- Gap < 10%: modelo generaliza bem
- Gap 10-20%: overfitting leve
- Gap > 20%: overfitting significativo


In [15]:
# PREDICOES TEMPORAIS - SERIE TEMPORAL

# Agregar predicoes por data
test_with_predictions = test_data.copy()
test_with_predictions['Predicted'] = best_results['y_pred_test']

# Vendas totais por dia
daily_comparison = test_with_predictions.groupby('Date').agg({
    'Sales': 'sum',
    'Predicted': 'sum'
}).reset_index()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily_comparison['Date'],
    y=daily_comparison['Sales'],
    mode='lines',
    name='Vendas Reais',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=daily_comparison['Date'],
    y=daily_comparison['Predicted'],
    mode='lines',
    name='Predicoes',
    line=dict(color='red', width=2, dash='dash')
))

fig.update_layout(
    title=f'Predicoes vs Vendas Reais ao Longo do Tempo - {best_model_name}',
    xaxis_title='Data',
    yaxis_title='Vendas Totais (EUR)',
    height=500,
    hovermode='x unified'
)
fig.write_html("../images/time_series_predictions.html")
fig.show()

# Calcular erro medio diario
daily_comparison['Error'] = daily_comparison['Sales'] - daily_comparison['Predicted']
daily_comparison['Error_Pct'] = (daily_comparison['Error'] / daily_comparison['Sales'] * 100)

print(f"\nERRO MEDIO DIARIO:")
print(f"   Erro Absoluto Medio: EUR {daily_comparison['Error'].abs().mean():,.2f}")
print(f"   Erro Percentual Medio: {daily_comparison['Error_Pct'].abs().mean():.2f}%")


ERRO MEDIO DIARIO:
   Erro Absoluto Medio: EUR 305,888.87
   Erro Percentual Medio: 12.55%


In [16]:
# ANALISE DE ERROS POR LOJA

# Calcular RMSE por loja
test_with_predictions = test_data.copy()
test_with_predictions['Predicted'] = best_results['y_pred_test']

store_performance = test_with_predictions.groupby('Store').apply(
    lambda x: pd.Series({
        'RMSE': np.sqrt(mean_squared_error(x['Sales'], x['Predicted'])),
        'MAE': mean_absolute_error(x['Sales'], x['Predicted']),
        'Mean_Sales': x['Sales'].mean(),
        'N_Predictions': len(x)
    })
).reset_index()

# Identificar melhores e piores lojas
top_10_stores = store_performance.nsmallest(10, 'RMSE')
bottom_10_stores = store_performance.nlargest(10, 'RMSE')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Top 10 Lojas (Menor RMSE)', 'Bottom 10 Lojas (Maior RMSE)')
)

fig.add_trace(
    go.Bar(x=top_10_stores['Store'].astype(str), y=top_10_stores['RMSE'],
           marker_color='lightgreen', name='Top 10'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=bottom_10_stores['Store'].astype(str), y=bottom_10_stores['RMSE'],
           marker_color='lightcoral', name='Bottom 10'),
    row=1, col=2
)

fig.update_xaxes(title_text="Loja", row=1, col=1)
fig.update_xaxes(title_text="Loja", row=1, col=2)
fig.update_yaxes(title_text="RMSE", row=1, col=1)
fig.update_yaxes(title_text="RMSE", row=1, col=2)

fig.update_layout(height=500, showlegend=False,
                  title_text="Performance por Loja")
fig.write_html("../images/store_performance.html")
fig.show()

print("\nRESUMO DE PERFORMANCE POR LOJA:")
print(f"   Melhor loja (menor RMSE): {top_10_stores.iloc[0]['Store']} - RMSE: {top_10_stores.iloc[0]['RMSE']:.2f}")
print(f"   Pior loja (maior RMSE): {bottom_10_stores.iloc[0]['Store']} - RMSE: {bottom_10_stores.iloc[0]['RMSE']:.2f}")
print(f"   RMSE medio entre lojas: {store_performance['RMSE'].mean():.2f}")


RESUMO DE PERFORMANCE POR LOJA:
   Melhor loja (menor RMSE): 48.0 - RMSE: 345.82
   Pior loja (maior RMSE): 909.0 - RMSE: 3616.56
   RMSE medio entre lojas: 750.51


In [17]:
# SALVAMENTO DO MELHOR MODELO

import pickle
import os

# Criar diretorio para modelos
os.makedirs('../models', exist_ok=True)

# Salvar modelo
model_filename = f"../models/best_model_{best_model_name.lower().replace(' ', '_')}.pkl"
with open(model_filename, 'wb') as f:
    pickle.dump(best_results['model'], f)

# Salvar label encoders
with open('../models/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

# Salvar metricas
metrics_summary = {
    'best_model': best_model_name,
    'test_metrics': best_results['metrics_test'],
    'train_metrics': best_results['metrics_train'],
    'all_models_comparison': comparison_df.to_dict('records')
}

with open('../models/metrics_summary.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)

print("\nMODELO E ARTEFATOS SALVOS COM SUCESSO!")
print("="*70)
print(f"Modelo: {model_filename}")
print("Label Encoders: ../models/label_encoders.pkl")
print("Metricas: ../models/metrics_summary.json")


MODELO E ARTEFATOS SALVOS COM SUCESSO!
Modelo: ../models/best_model_xgboost.pkl
Label Encoders: ../models/label_encoders.pkl
Metricas: ../models/metrics_summary.json


In [18]:
# RESUMO FINAL E CONCLUSOES

print("\n" + "="*70)
print("RESUMO FINAL DO PROJETO")
print("="*70)

print("\n1. DADOS:")
print(f"   - Treino: {len(train_data):,} registros")
print(f"   - Teste: {len(test_data):,} registros")
print(f"   - Features: {len(all_features)}")
print(f"   - Periodo de teste: {test_data['Date'].min().date()} a {test_data['Date'].max().date()}")

print("\n2. MODELOS TREINADOS:")
for i, row in comparison_df.iterrows():
    print(f"   {i+1}. {row['Modelo']}")
    print(f"      - RMSE: {row['RMSE']:,.2f}")
    print(f"      - R2: {row['R2']:.4f}")
    print(f"      - MAPE: {row['MAPE (%)']:.2f}%")

print(f"\n3. MELHOR MODELO: {best_model_name}")
print(f"   - RMSE no teste: EUR {best_rmse:,.2f}")
print(f"   - R2: {best_r2:.4f} ({best_r2*100:.2f}% da variancia explicada)")
print(f"   - MAPE: {comparison_df.iloc[0]['MAPE (%)']:.2f}%")

print("\n4. TOP 5 FEATURES MAIS IMPORTANTES:")
for i, row in importance_df.head(5).iterrows():
    print(f"   {i+1}. {row['Feature']}")

print("\n5. PROXIMOS PASSOS:")
print("   - Deploy do modelo em producao")
print("   - Monitoramento continuo de performance")
print("   - Retraining periodico com novos dados")
print("   - Implementacao de API para predicoes em tempo real")
print("   - Dashboard de visualizacao de predicoes")

print("\n" + "="*70)
print("NOTEBOOK 02 CONCLUIDO COM SUCESSO!")
print("="*70)


RESUMO FINAL DO PROJETO

1. DADOS:
   - Treino: 802,014 registros
   - Teste: 199,585 registros
   - Features: 30
   - Periodo de teste: 2015-02-03 a 2015-07-31

2. MODELOS TREINADOS:
   1. XGBoost
      - RMSE: 796.13
      - R2: 0.9572
      - MAPE: 8.79%
   2. LightGBM
      - RMSE: 804.58
      - R2: 0.9563
      - MAPE: 8.92%
   3. Random Forest
      - RMSE: 808.58
      - R2: 0.9558
      - MAPE: 8.78%
   4. Linear Regression
      - RMSE: 1,379.37
      - R2: 0.8715
      - MAPE: 14.35%

3. MELHOR MODELO: XGBoost
   - RMSE no teste: EUR 796.13
   - R2: 0.9572 (95.72% da variancia explicada)
   - MAPE: 8.79%

4. TOP 5 FEATURES MAIS IMPORTANTES:
   3. Open
   5. StateHoliday
   27. sales_lag_14
   4. Promo
   29. sales_rolling_30

5. PROXIMOS PASSOS:
   - Deploy do modelo em producao
   - Monitoramento continuo de performance
   - Retraining periodico com novos dados
   - Implementacao de API para predicoes em tempo real
   - Dashboard de visualizacao de predicoes

NOTEBOOK 02 C